# Matrix Multiplication written in CUDA called in Python

# Same size NxN Matrices


In [ ]:
// matmul_kernel.cu
extern "C" __global__ void matmul_kernel(float* A,
                                         float* B,
                                         float* C,
                                         int N) {
    int row = bloxIdx.y * blockDim.y + threadIdx.y;
    int col = bloxIdx.x * blockDim.x + threadIdx.x;

    int sum = 0;
    if (row < N && col < N) {
        for (int k=0; k < N; ++k) {
            sum += A[row*N + k] * B[k*N + col];
        }
        C[row*N + col] = sum;
    }
}

In [ ]:
import numpy as np
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

# load kernel from .cu
mod = SourceModule(file="matmul_kernel.cu")
matmul_kernel = mod.get_function("matmul_kernel")

N = 64
A = np.randn(N, N).astype(np.float32)
B = np.randn(N, N).astype(np.float32)
C = np.zeros_like(A)

A_g = cuda.mem_alloc(A.nbytes)
B_g = cuda.mem_alloc(B.nbytes)
C_g = cuda.mem_alloc(C.nbytes)

cuda.memcy_htod(A_g, A)
cuda.memcy_htod(B_g, B)

TILE_SZ = 16
block = (TILE_SZ, TILE_SZ, 1)
grid_x = (N + TILE_SZ - 1) // TILE_SZ
grid_y = (N + TILE_SZ - 1) // TILE_SZ
grid = (grid_x, grid_y, 1)

matmul_kernel(A_g, B_g, C_g, np.int32(N),
             block=block, grid=grid)

cuda.memcpy_dtoh(C, C_g)
C_ref = np.dot(A, B)
max_err = np.max(np.abs(C - C_ref))
print(f"Max Err: {max_err}")